# AI Job Search Assistant Agent using LangGraph + Groq + Google Search + Yagmail

This AI agent:
- Finds real-time job postings
- Analyzes job relevance
- Shortlists best opportunities

Tech Stack:
- LangGraph
- Groq LLM
- SerpAPI (Google Search)
- Yagmail
- Python

In [ ]:
%pip install langgraph

In [ ]:
%pip install langchain

In [ ]:
%pip install langchain-groq

In [ ]:
%pip install google-search-results

In [ ]:
%pip install yagmail

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph
from langchain_groq import ChatGroq
import yagmail
from serpapi import GoogleSearch

In [ ]:
GROQ_API_KEY = "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"
SERP_API_KEY = "e5ca9a5494ffa2702da4621a19c3d5db4e96ad03ed220dbd771782fb856c9714"

In [ ]:
yag = yagmail.SMTP(
    "sairaashraf95134@gmail.com",
    "hxfy cmab jzwn dagh"
)

In [ ]:
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send job list or CV to recruiter/user."""

    yag.send(
        to=recipient,
        subject=subject,
        contents=body
    )

    return "Email sent successfully!"

In [ ]:
def search_jobs_tool(query: str) -> str:

    params = {
        "engine": "google_jobs",
        "q": query,
        "api_key": SERP_API_KEY
    }

    search = GoogleSearch(params)

    results = search.get_dict()

    jobs_results = results.get("jobs_results", [])

    if not jobs_results:
        return "No jobs found."

    jobs_text = ""

    for job in jobs_results[:5]:

        title = job.get("title", "No Title")

        company = job.get("company_name", "No Company")

        location = job.get("location", "No Location")

        description = job.get("description", "No Description")


        jobs_text += f"""
Job Title: {title}

Company: {company}

Location: {location}

Description: {description}

-----------------------------------
"""

    return jobs_text

In [ ]:
print(
    search_jobs_tool(
        "Python Developer Remote"
    )
)

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY
)

In [ ]:
class AgentState(TypedDict):

    message: str

In [ ]:
def job_agent(state):

    user_message = state["message"]

    print("USER REQUEST:")
    print(user_message)

    jobs = search_jobs_tool(user_message)

    ai_response = llm.invoke(
        f"""
        You are a professional AI Job Assistant.

        Analyze and organize these jobs professionally.

        Jobs:
        {jobs}
        """
    )

    final_jobs = ai_response.content

    send_email_tool(
        "sairaashraf95134@gmail.com",
        "Latest AI/Python Jobs",
        final_jobs
    )

    return {

        "message": final_jobs

    }

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("agent", job_agent)

graph.set_entry_point("agent")

graph.set_finish_point("agent")

app = graph.compile()

In [ ]:
from IPython.display import Markdown, display
result = app.invoke({

    "message": "Python Developer Remote Jobs"

})

display(Markdown(result["message"]))